# NB-07-01 — Neural Network Fundamentals Workbench

**Module M07 · Level 300**

This notebook is the computational companion to lessons L07-01 to L07-04. Everything here is
implemented in NumPy (`nn_numpy.py`): activations, a multilayer perceptron, backpropagation,
mini-batch training and four optimisers.

By the end you will have:

1. Seen a single perceptron solve AND and OR, and fail on XOR.
2. Trained an MLP that solves XOR.
3. Verified backpropagation numerically against finite differences.
4. Compared SGD, Momentum, RMSProp and Adam on the same problem.
5. Watched loss curves and diagnosed an overfitting run.

Datasets come from `deep_data.py` and are synthetic, seeded and CPU-safe; see the module README
for the public datasets they stand in for.

In [1]:
import os, sys, math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Locate the module's code directory whether Jupyter starts here or at the repo root.
for candidate in ("../code", "code", os.path.join("modules", "M07-neural-networks-and-deep-learning", "code")):
    if os.path.isdir(candidate):
        CODE_DIR = os.path.abspath(candidate)
        break
else:
    raise RuntimeError("Run this notebook from the module's notebooks/ directory")
sys.path.insert(0, CODE_DIR)

import deep_data as dd
import nn_numpy
from nn_numpy import MLP, Adam, Momentum, RMSProp, SGD, accuracy, gradient_check, perceptron_rule, train
from torch_utils import (Trainer, SmallCNN, LSTMRegressor, TinyTransformerClassifier, AutoEncoder,
                         count_parameters, freeze, unfreeze, get_device, make_loader, set_seed,
                         load_checkpoint, save_checkpoint)
import transfer

torch.set_num_threads(2)
set_seed(0)
print("code:", CODE_DIR)
print("torch:", torch.__version__, "| device:", get_device())

code: C:\Users\Nadeem\Documents\Artificial-Intelligence-Course\modules\M07-neural-networks-and-deep-learning\code
torch: 2.14.1+cpu | device: cpu


## 1. A single perceptron on logic gates

A perceptron is one neuron with a step activation: it draws a straight line through the input
space. AND and OR are linearly separable, so the perceptron rule converges. XOR is not, so it
cannot.

In [2]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
print("inputs:\n", X)

gates = {
    "AND": np.array([0, 0, 0, 1]),
    "OR": np.array([0, 1, 1, 1]),
    "XOR": np.array([0, 1, 1, 0]),
}
for name, labels in gates.items():
    w, b, errors = perceptron_rule(X, labels, epochs=6, lr=1.0)
    preds = ((X @ w + b) > 0).astype(int)
    print(f"{name:4s} errors per epoch {errors} final acc {accuracy(labels, preds):.2f}")

inputs:
 [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
AND  errors per epoch [1, 3, 3, 2, 1, 0] final acc 1.00
OR   errors per epoch [1, 2, 1, 0, 0, 0] final acc 1.00
XOR  errors per epoch [2, 3, 4, 4, 4, 4] final acc 0.50


The XOR row never reaches zero errors: its two diagonal classes need two lines, so one neuron is
provably insufficient. This is the historical motivation for hidden layers.

## 2. An MLP solves XOR

One hidden layer of four neurons is enough. Training is plain mini-batch gradient descent through
backpropagation.

In [3]:
xor_x, xor_y = dd.xor_data()
xor_y = xor_y.ravel().astype(int)

model = MLP([2, 4, 2], activation="tanh", seed=1)
history = train(model, xor_x, xor_y, epochs=2000, batch_size=4, optimizer=Adam(0.05), seed=1)
print("final loss:", f"{history['loss'][-1]:.2e}")
print("accuracy:", round(accuracy(xor_y, model.predict(xor_x)), 3))
print("predictions:", model.predict(xor_x), "expected:", xor_y)

final loss: 5.95e-05
accuracy: 1.0
predictions: [0 1 1 0] expected: [0 1 1 0]


The loss falls by several orders of magnitude. Note the exact output formatting may differ from the
lesson text depending on the seed, but accuracy reaches 1.0.

## 3. Verifying backpropagation

Backpropagation is easy to get subtly wrong. The standard defence is a numerical gradient check:
compare the analytic gradient with a central finite difference on a handful of parameters.

In [4]:
check_model = MLP([64, 5, 3], activation="tanh", seed=0)
Xc, yc = dd.digit_images(60, size=8, seed=1)
max_rel_error = gradient_check(check_model, Xc.reshape(len(Xc), -1), yc % 3, epsilon=1e-6, seed=0)
print("maximum relative error:", f"{max_rel_error:.2e}")
print("verdict:", "PASS (below 1e-6)" if max_rel_error < 1e-6 else "FAIL - recheck the chain rule")

maximum relative error: 1.22e-08
verdict: PASS (below 1e-6)


## 4. Optimisers on the same problem

Identical architecture, identical data, identical seed. Only the update rule changes. We train for
five epochs and report loss and accuracy.

In [5]:
Xo, yo = dd.digit_images(1200, size=8, seed=0)
split = 900

rows = []
for name, opt in [("sgd", SGD(0.05)),
                  ("momentum", Momentum(0.05, beta=0.9)),
                  ("rmsprop", RMSProp(0.01)),
                  ("adam", Adam(0.01))]:
    net = MLP([64, 32, 10], activation="relu", seed=0)
    hist = train(net, Xo[:split].reshape(split, -1), yo[:split], epochs=5, batch_size=64,
                 optimizer=opt, seed=0)
    test_acc = accuracy(yo[split:], net.predict(Xo[split:].reshape(-1, 64)))
    rows.append((name, hist["loss"][-1], test_acc))

print(f"{'optimiser':10s} {'train loss':>10s} {'test acc':>9s}")
for name, loss, acc in rows:
    print(f"{name:10s} {loss:10.3f} {acc:9.3f}")

optimiser  train loss  test acc
sgd             1.834     0.540
momentum        0.263     0.957
rmsprop         0.085     0.997
adam            0.099     0.990


Five epochs is deliberately short: Momentum and Adam make rapid early progress, while plain SGD
needs many more steps. Longer runs narrow the gap — the lesson's point is that optimiser choice is a
speed and stability choice, not a free accuracy win.

## 5. Overfitting, and what dropout does to it

Train a generous network on a small noisy training set, then compare with the same network using
dropout.

In [6]:
Xr, yr = dd.digit_images(1000, size=8, seed=2)
rng = np.random.default_rng(0)
noisy = yr.copy()
flip = rng.random(len(noisy)) < 0.25
noisy[flip] = rng.integers(0, 10, flip.sum())
tr = 700

for dropout in (0.0, 0.3):
    net = MLP([64, 64, 32, 10], activation="relu", seed=0)
    hist = train(net, Xr[:tr].reshape(tr, -1), noisy[:tr], epochs=25, batch_size=32,
                 optimizer=Adam(0.01), dropout=dropout, seed=0)
    train_acc = accuracy(noisy[:tr], net.predict(Xr[:tr].reshape(tr, -1)))
    test_acc = accuracy(yr[tr:], net.predict(Xr[tr:].reshape(-1, 64)))
    print(f"dropout {dropout:.1f}: train acc {train_acc:.3f} clean test acc {test_acc:.3f}")

dropout 0.0: train acc 0.976 clean test acc 0.827


dropout 0.3: train acc 0.833 clean test acc 0.957


Without dropout the network memorises the flipped labels: training accuracy 0.976 while clean test
accuracy is only 0.827. Dropout trades training accuracy for generalisation — 0.833 on the noisy
training labels, 0.957 on clean test data. That gap, not the absolute accuracy, is the overfitting
signal to watch.

## 6. Exercise

1. Change the hidden width to 2 and re-run XOR. Why does it fail?
2. Run the optimiser comparison for 30 epochs. Does the ordering change?
3. Sweep dropout over `0.0, 0.1, 0.3, 0.5` and plot clean test accuracy.

**References:** L07-01 to L07-04. Code: `code/nn_numpy.py`, `code/deep_data.py`.